In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install tqdm

In [ ]:
# @title Manga Upscaler + Downscaler (Per-Image Timer) { display-mode: "form" }

import os
from google.colab import drive
import subprocess
from pathlib import Path
import torch
from PIL import Image
import shutil
import time

# ==== ✅ CUSTOM FOLDERS ====
BW_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN (1)/bw/Vol.01 Ch.0002 - Rope Partner (en) [Bakana Haven]" # @param {type:"string"}
COLOR_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN/color" # @param {type:"string"}
FINAL_OUTPUT_FOLDER = "/content/gdr" # @param {type:"string"}
# ==== CORE SETUP ====
def check_connect_gdrive():
    if not os.path.exists("/content/gdrive/MyDrive"):
        print("🔌 Mounting Google Drive...")
        drive.mount("/content/gdrive")

def check_clone_esrgan():
    if not os.path.exists("ESRGAN"):
        print("⬇️ Cloning ESRGAN...")
        subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"])

def init_dirs():
    Path(FINAL_OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    Path("/content/temp_input").mkdir(parents=True, exist_ok=True)
    Path("/content/temp_output").mkdir(parents=True, exist_ok=True)

def dir_contains_files(path):
    return os.path.exists(path) and any(Path(path).glob("*"))

def upscale_and_downscale_image(image_path, model_path):
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"

    # Clear temp folders
    for folder in [temp_input, temp_output]:
        for file in Path(folder).glob("*"):
            file.unlink()

    # Copy image to temp input
    shutil.copy(image_path, temp_input)

    # Run ESRGAN on the image
    subprocess.run([
        "python", "ESRGAN/upscale.py", "-se",
        "-i", temp_input,
        "-o", temp_output,
        model_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Resize immediately
    filename = os.path.basename(image_path)
    upscaled_path = os.path.join(temp_output, filename)
    final_output_path = os.path.join(FINAL_OUTPUT_FOLDER, filename)

    if os.path.exists(upscaled_path):
        with Image.open(upscaled_path) as img:
            new_size = (img.width // 2, img.height // 2)
            resized = img.resize(new_size, Image.LANCZOS)
            resized.save(final_output_path)
            return True
    return False

def process_folder(folder_path, model_path):
    print(f"\n📁 Processing: {folder_path}")
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    images = [f for f in os.listdir(folder_path) if Path(f).suffix.lower() in exts]

    total = len(images)
    for idx, file in enumerate(images, 1):
        image_path = os.path.join(folder_path, file)
        start = time.time()

        success = upscale_and_downscale_image(image_path, model_path)

        end = time.time()
        duration = end - start
        if success:
            print(f"✅ {file} ({idx}/{total}) | ⏱ {duration:.2f} sec")
        else:
            print(f"❌ Failed: {file} ({idx}/{total})")

def main():
    print("📈 Manga Upscaler + 2x Downscaler with Per-Image Timing")

    if not torch.cuda.is_available():
        print("❌ GPU not enabled. Set 'Runtime' → 'Change runtime type' → GPU.")
        return

    check_connect_gdrive()
    check_clone_esrgan()
    init_dirs()

    if dir_contains_files(BW_INPUT_FOLDER):
        print("🎨 Upscaling B&W images...")
        process_folder(BW_INPUT_FOLDER, "ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth")

    if dir_contains_files(COLOR_INPUT_FOLDER):
        print("🎨 Upscaling Color images...")
        process_folder(COLOR_INPUT_FOLDER, "ESRGAN/models/4x-AnimeSharp.pth")

    print(f"\n📁 Done. Final output saved to:\n{FINAL_OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()


In [ ]:
# @title Manga Upscaler (Retry Failed Images) { display-mode: "form" }

import os
import shutil
import time
from pathlib import Path
from google.colab import drive
import subprocess
import torch

# ==== ✅ CUSTOM FOLDERS ====
BW_INPUT_FOLDER = "/content/gdrive/MyDrive/ESRGAN (1)/bw"  # @param {type:"string"}
COLOR_INPUT_FOLDER = ""  # @param {type:"string"}
FINAL_OUTPUT_FOLDER = "/content/render_tempt"  # @param {type:"string"}

# ==== SETUP ====
def mount_drive():
    if not os.path.exists("/content/gdrive/MyDrive"):
        print("🔌 Mounting Google Drive...")
        drive.mount("/content/gdrive")

def clone_esrgan():
    if not os.path.exists("ESRGAN"):
        print("⬇️ Cloning ESRGAN...")
        subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"], check=True)

def prepare_dirs():
    os.makedirs(FINAL_OUTPUT_FOLDER, exist_ok=True)
    os.makedirs("/content/temp_input", exist_ok=True)
    os.makedirs("/content/temp_output", exist_ok=True)

def get_all_images(folder):
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in exts]

def upscale_image(image_path, model_path, rel_subfolder):
    temp_input = "/content/temp_input"
    temp_output = "/content/temp_output"

    # Clean temp
    for folder in [temp_input, temp_output]:
        for file in Path(folder).glob("*"):
            file.unlink()

    # Copy input
    shutil.copy(image_path, temp_input)

    # Run ESRGAN
    result = subprocess.run([
        "python", "ESRGAN/upscale.py", "-se",
        "-i", temp_input,
        "-o", temp_output,
        model_path
    ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    filename = image_path.name
    upscaled_path = os.path.join(temp_output, filename)
    out_dir = os.path.join(FINAL_OUTPUT_FOLDER, rel_subfolder)
    os.makedirs(out_dir, exist_ok=True)
    final_path = os.path.join(out_dir, filename)

    if os.path.exists(upscaled_path):
        shutil.copy(upscaled_path, final_path)
        return True
    else:
        return False

def process_folder(folder_path, model_path, label=""):
    print(f"\n📁 Processing {label} recursively: {folder_path}")
    images = get_all_images(folder_path)
    total = len(images)
    failed_images = []

    for idx, img_path in enumerate(images, 1):
        rel_subfolder = str(img_path.parent.relative_to(folder_path))
        start = time.time()
        success = upscale_image(img_path, model_path, rel_subfolder)
        end = time.time()

        if success:
            print(f"✅ {img_path.name} ({idx}/{total}) | ⏱ {end - start:.2f}s")
        else:
            print(f"❌ Failed: {img_path.name} ({idx}/{total})")
            failed_images.append((img_path, rel_subfolder))

    # === Retry failed images ===
    if failed_images:
        print(f"\n🔁 Retrying {len(failed_images)} failed image(s)...")
        retry_failed = []
        for i, (img_path, rel_subfolder) in enumerate(failed_images, 1):
            print(f"🔁 Retrying {img_path.name} ({i}/{len(failed_images)})")
            success = upscale_image(img_path, model_path, rel_subfolder)
            if success:
                print(f"✅ Success on retry: {img_path.name}")
            else:
                print(f"❌ Still failed: {img_path.name}")
                retry_failed.append(img_path)

        if retry_failed:
            print(f"\n🚨 {len(retry_failed)} image(s) failed after retry:")
            for path in retry_failed:
                print(f" - {path}")
        else:
            print("\n🎉 All previously failed images succeeded on retry!")
    else:
        print("\n✅ No failed images to retry.")

def main():
    print("🚀 Manga Upscaler (Retry Failed Images)")
    if not torch.cuda.is_available():
        print("❌ Please enable GPU: Runtime → Change Runtime Type → GPU")
        return

    mount_drive()
    clone_esrgan()
    prepare_dirs()

    if get_all_images(BW_INPUT_FOLDER):
        print("🖤 Upscaling B&W images...")
        process_folder(BW_INPUT_FOLDER, "ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth", label="B&W")

    if get_all_images(COLOR_INPUT_FOLDER):
        print("🎨 Upscaling Color images...")
        process_folder(COLOR_INPUT_FOLDER, "ESRGAN/models/4x-AnimeSharp.pth", label="Color")

    print(f"\n📦 Done! Upscaled images are in:\n👉 {FINAL_OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()


In [ ]:
# @title Downscaler (Per-Image Timer)

import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms.functional as TF
from concurrent.futures import ThreadPoolExecutor

# Config
input_folder = "/content/gdrive/MyDrive/ESRGAN (1)/output" # @param {type:"string"}
output_folder = "/content/gdrive/MyDrive/ESRGAN/output1" # @param {type:"string"}
batch_size = 16
max_workers = 4  # parallel threads (try 4–8 depending on CPU)

exts = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# Step 1: Collect all image paths
image_paths = []
for root, _, files in os.walk(input_folder):
    for file in files:
        if Path(file).suffix.lower() in exts:
            image_paths.append(os.path.join(root, file))

# Step 2: Batch the paths
def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# Step 3: GPU resize logic per batch
def process_batch(batch_paths):
    tensors = []
    rel_paths = []

    # Load images
    for path in batch_paths:
        try:
            with Image.open(path).convert("RGB") as img:
                tensor = TF.to_tensor(img)
                tensors.append(tensor)
                rel_paths.append(os.path.relpath(path, input_folder))
        except Exception as e:
            print(f"❌ Failed to load: {path} — {e}")

    if not tensors:
        return

    # Stack and move to GPU
    try:
        batch_tensor = torch.stack(tensors).to(device)
        _, _, h, w = batch_tensor.shape
        new_size = (h // 2, w // 2)

        # GPU resize
        resized = torch.nn.functional.interpolate(batch_tensor, size=new_size, mode='bicubic', align_corners=False)

        # Save images
        for i, rel_path in enumerate(rel_paths):
            out_path = os.path.join(output_folder, rel_path)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            img_out = TF.to_pil_image(resized[i].cpu())
            img_out.save(out_path)
    except Exception as e:
        print(f"⚠️ Error during GPU resize: {e}")

# Step 4: Run all batches in parallel
batches = list(chunk(image_paths, batch_size))

print(f"📦 Found {len(image_paths)} images in {len(batches)} batches.")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(executor.map(process_batch, batches), total=len(batches), desc="⚡ Resizing", ncols=80))

print("✅ All images processed with GPU + parallel I/O.")


# AI Manga Upscale Colab GITHUB



In [ ]:
# @title Manga Upscaler { display-mode: "form" }

import os
from google.colab import drive
import subprocess
from pathlib import Path
import torch

def check_connect_gdrive():
  if not os.path.exists("/content/gdrive/MyDrive"):
    print("Google Drive connection in progress...")
    drive.mount("/content/gdrive")

def check_clone_esrgan():
  if not os.path.exists("ESRGAN"):
    print("Downloading ESRGAN along with the AI models...")
    subprocess.run(["git", "clone", "https://github.com/Spladenly/ESRGAN"])
def init_dirs():
  Path("/content/gdrive/MyDrive/ESRGAN").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/bw").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/color").mkdir \
   (parents=True, exist_ok=True)
  Path("/content/gdrive/MyDrive/ESRGAN/output").mkdir \
   (parents=True, exist_ok=True)

def dir_contains_files(path):
  for root, dirs, files in os.walk(path):
    if files:
      return True
  return False

def ai_process_bw():
  !python ESRGAN/upscale.py -se -i /content/gdrive/MyDrive/ESRGAN/bw \
  -o /content/gdrive/MyDrive/ESRGAN/output \
  ESRGAN/models/4x_eula_digimanga_bw_v2_nc1_307k.pth

def ai_process_color():
  !python ESRGAN/upscale.py -se -i /content/gdrive/MyDrive/ESRGAN/color \
  -o /content/gdrive/MyDrive/ESRGAN/output \
  ESRGAN/models/4x-AnimeSharp.pth

def main():
  print("[AI Manga Upscale Colab] Manga Upscaler")

  if not torch.cuda.is_available():
    print("This session doesn't have a GPU.\n To connect a GPU, click:\n'Edit' -> 'Notebook settings' -> 'Hardware accelerator' = GPU; 'GPU type' = T4.\nAfter that, run this script again.")
    return

  check_connect_gdrive()
  check_clone_esrgan()
  init_dirs()

  status = 0

  if dir_contains_files("/content/gdrive/MyDrive/ESRGAN/bw"):
    status += 1
    print("Upscaling bw...")
    ai_process_bw()

  if dir_contains_files("/content/gdrive/MyDrive/ESRGAN/color"):
    status += 1
    print("Upscaling color...")
    ai_process_color()

  if status == 0:
    print("No pages were found in the following directories: '/ESRGAN/bw' and '/ESRGAN/color' on your Google Drive.\nPlease upload manga pages there, and run this script again.")
  else:
    print("The processing has been finished. The result can be downloaded from '/ESRGAN/output' on your Google Drive.\nTo process additional pages, run this script again. If you don't plan to process additional pages in the near future, please close the current session.\nThis can be done by clicking on: the inverted triangle (next to the 'RAM' and 'Disk' labels in the upper right corner) -> 'Disable and remove runtime'.\nThis will unlock the resources reserved for this session for other users.")

if __name__ == "__main__":
  main()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

You may also upload CBZs and ZIPs instead of images. Use the following script to extract them before running Upscale Manga. After the extraction, the original files will be moved to recycle bin.

In [ ]:
# @title Extractor { display-mode: "form" }

import os
import shutil
import zipfile
from google.colab import drive

def check_connect_gdrive():
  if not os.path.exists("/content/gdrive/MyDrive"):
    print("Google Drive connection in progress...")
    drive.mount("/content/gdrive")

def unpack(path):
  zip_files = [f for f in os.listdir(path) if f.endswith(".zip") or \
               f.endswith(".cbz")]

  print(f"Found {len(zip_files)} file(s).")

  for zip_file in zip_files:
      print(f"Extracting {zip_file}...")

      zip_path = os.path.join(path, zip_file)

      if not os.path.exists(os.path.splitext(zip_path)[0]):
        extract_folder = os.path.join(path, \
                                        os.path.splitext(zip_file)[0])
        os.makedirs(extract_folder, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_folder)

        os.remove(zip_path)
      else:
         print("The folder already exists -- skipping.")

def main():
  print("[AI Manga Upscale Colab] Extractor")

  check_connect_gdrive()

  status = 0
  bw_path = "/content/gdrive/MyDrive/ESRGAN/bw"
  color_path = "/content/gdrive/MyDrive/ESRGAN/color"

  if os.path.exists(bw_path):
    status += 1
    print(f"Scanning: {bw_path}")
    unpack(bw_path)

  if os.path.exists(color_path):
    status += 1
    print(f"Scanning: {color_path}")
    unpack(color_path)

  if status == 0:
    print("Both 'bw' and 'color' directories don't exist.")

  print("Done.")

if __name__ == "__main__":
  main()